# 09 — Repair: download again the versions whose files never reached Drive

Owner: Member 1. **Run once on Google Colab (CPU runtime), before the full Milestone 2 run.**

A check of the crawl (11 Nov 2026) found 1,273 versions (1,104 papers, all in `2609.34965`–`2609.36365`)
whose receipt in `state/download/receipts.jsonl` says `ok` but whose `data/raw/<id>/vN/` folder is not on
Drive: probably a Colab session that ended before Google Drive finished syncing `data/`, while the receipts
in `state/` were already synced. `05` trusts the receipts (`DOWNLOAD_VERIFY = "receipt"`), so it never asks
for these versions again, and Milestone 2 would record them as `empty_source`.

What this notebook does, with the code of `02_download` unchanged:

1. **Find** every `ok` receipt whose `vN/` folder is missing on Drive (one folder check per version, 16 threads).
2. **Download** those papers again with `download_sources` (one connection, `ARXIV_DELAY` = 3 s apart, 429
   handling, progress bars), into the same `data/raw/`. Its state goes to a separate folder,
   `state/download_repair/`, so `state/download/` (receipts, `papers.csv`) and the Milestone 1 logs are not
   touched: the main receipts already describe these files.
3. **Check** again that every folder is now there, then flush Google Drive.

**Time:** about 3 s per version, so roughly an hour for 1,273 versions (more if arXiv answers HTTP 429).
**Resume:** after a disconnect, run all cells again; step 1 then finds only what is still missing. Do not
run `05` or `06` at the same time (same `pipeline.lock`). If the download stops with *arXiv refused ... in a
row*, use **Runtime > Disconnect and delete runtime** and run all again (the IP is rate-limited).

In [ ]:
# On Colab: clone or update the branch, install packages and work from src/ (the notebooks use relative paths).
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if ON_COLAB:
    REPO = Path("/content/intro-ds-arxiv-project")
    BRANCH = "Dedup"
    URL = "https://github.com/thinhelzan842/intro-ds-arxiv-project.git"
    if not (REPO / ".git").is_dir():
        subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", URL, str(REPO)], check=True)
    else:   # explicit refspec: also works in a clone of another single branch (e.g. VA, cloned by 05)
        subprocess.run(["git", "-C", str(REPO), "fetch", "origin",
                        f"+refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}"], check=True)
        subprocess.run(["git", "-C", str(REPO), "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "src/requirements.txt")], check=True)
    os.chdir(REPO / "src")
    print("Code in use:", subprocess.run(["git", "-C", str(REPO), "log", "--oneline", "-1"],
                                         capture_output=True, text=True).stdout.strip())

RUN_TEST = False
CHECK_ONLY = False          # True: only list the missing versions (step 1), download nothing
FLUSH_DRIVE_AT_END = True   # flush and unmount Google Drive at the end, so every file is saved
globals().update(globals().get("REPAIR_OVERRIDES", {}))

get_ipython().run_line_magic("run", "-i ./00_config.ipynb")
get_ipython().run_line_magic("run", "-i ./02_download.ipynb")

from collections import Counter
from concurrent.futures import ThreadPoolExecutor

REPAIR_STATE = STATE_DIR / "download_repair"
REPAIR_MANIFEST = REPAIR_STATE / "manifest.json"

## 1. Versions with an `ok` receipt but no folder on Drive

In [ ]:
def missing_versions():
    """Sorted [(arxiv_id, version)] whose receipt is ok but whose vN/ folder does not exist."""
    receipts = _read_receipt_index(DOWNLOAD_STATE)
    ok = sorted(key for key, receipt in receipts.items() if receipt.get("status") == "ok")
    exists = lambda key: (DATA_DIR / to_folder_id(key[0]) / f"v{key[1]}").is_dir()   # noqa: E731
    with ThreadPoolExecutor(16) as pool:
        found = list(tqdm(pool.map(exists, ok), total=len(ok), unit="version", desc="Checking")
                     if tqdm else pool.map(exists, ok))
    missing = [key for key, there in zip(ok, found) if not there]
    print(f"{len(ok):,} versions with an ok receipt; {len(missing):,} without a folder on Drive "
          f"({len({i for i, _ in missing}):,} papers)")
    return missing


MISSING = missing_versions()
if MISSING:
    print("first:", [f"{i}v{v}" for i, v in MISSING[:5]], "| last:", [f"{i}v{v}" for i, v in MISSING[-5:]])

## 2. Download them again

The manifest lists the affected papers with their number of versions (from `state/discovery_manifest.json`),
so `download_sources` asks for every version of those papers; a version that was on Drive is downloaded
again and replaced by the same files. The pipeline lock keeps `05`/`06` out while this runs.

In [ ]:
summary = None
if MISSING and not CHECK_ONLY:
    versions = {p["id"]: p["versions"] for p in read_json(STATE_DIR / "discovery_manifest.json")["papers"]}
    papers = sorted({arxiv_id for arxiv_id, _ in MISSING})
    REPAIR_STATE.mkdir(parents=True, exist_ok=True)
    write_json(REPAIR_MANIFEST, {"papers": [{"id": i, "versions": versions[i]} for i in papers]})
    print(f"Repair manifest: {len(papers):,} papers, {sum(versions[i] for i in papers):,} versions -> {REPAIR_MANIFEST}")
    lock = RunLock(PROJECT_ROOT / "pipeline.lock").acquire()
    try:
        summary = download_sources(REPAIR_MANIFEST, DATA_DIR, REPAIR_STATE, user_agent=DOWNLOAD_USER_AGENT,
                                   workers=DOWNLOAD_WORKERS, verify="receipt", progress=True)
    finally:
        lock.release()
    print({k: summary[k] for k in ("elapsed_seconds", "versions_ok", "versions_empty", "versions_failed_this_run",
                                    "stopped_reason")})

## 3. Check and flush Drive

Lists what is still missing (run all again to retry it; `state/download_repair/errors.jsonl` says why), then
flushes Google Drive, whose writes are asynchronous.

In [ ]:
still_missing = missing_versions() if summary else MISSING
if still_missing:
    print("Still missing:", [f"{i}v{v}" for i, v in still_missing[:20]], "..." if len(still_missing) > 20 else "")
    errors = REPAIR_STATE / "errors.jsonl"
    if errors.is_file():
        kinds = Counter(json.loads(line).get("error_kind") for line in errors.read_text(encoding="utf-8").splitlines() if line)
        print("Errors of the repair run(s):", dict(kinds))
else:
    print("Every version with an ok receipt has its folder on Drive.")

if IN_COLAB and FLUSH_DRIVE_AT_END:
    from google.colab import drive
    drive.flush_and_unmount()
    print("Google Drive flushed and unmounted: every file of this session is saved.")